# Gradio Web Application for Adapter-Tuned (LoRA) Model
## Objective
The objective of this notebook is to test and launch a live, interactive web application for the Adapter-Tuned (LoRA) NLLB translation model. It contains the complete, self-contained Python code required to build a Gradio-based user interface that allows users to perform bidirectional Odia-German translation.

## Methodology
The script is designed to be a complete web application that showcases the unique loading process for a PEFT model.

1. **Model Loading:** It first loads the original, large **base NLLB model** in 8-bit precision. It then loads the small, fine-tuned **LoRA adapters** from their repository on the Hugging Face Hub and applies them to the base model.
2. **Language Detection:** It implements a robust, hybrid language detection system, prioritizing a script-based check for Odia and using the `langdetect` library as a fallback.
3. **Translation Logic:** It defines a central `translate_text` function that takes user input, runs the detection logic, and calls the loaded model to generate the translation.
4. **Web Interface:** It uses the `gradio` library to create a clean user interface, complete with text boxes, a dropdown for manual language selection, and example sentences.

## Workflow
1. Installs all required libraries (`gradio`, `transformers`, `peft`, etc.).
2. Loads the base NLLB model and the LoRA adapters from the Hub.
3. Defines the language detection and translation functions.
4. Creates the Gradio `Interface` object.
5. Launches the web application, creating a temporary public URL for testing in the Colab environment.

## Input & Output
* **Input:** Text entered by a user into the Gradio web interface.
* **Output:** A live, interactive Gradio web application for bidirectional Odia-German translation powered by the LoRA fine-tuned model.

In [1]:
# Uninstall all potentially conflicting packages
!pip uninstall -y torch torchvision torchaudio transformers gradio langdetect sentencepiece accelerate huggingface-hub safetensors peft bitsandbytes torchtune sentence-transformers timm

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Succe

In [2]:
# Clear pip cache
!pip cache purge

Files removed: 0


In [3]:
# Install libraries
!pip install torch==2.9.0
!pip install transformers==4.57.3
!pip install gradio==5.50.0
!pip install langdetect==1.0.9
!pip install sentencepiece==0.2.1
!pip install huggingface-hub==0.36.0
!pip install accelerate==1.12.0
!pip install safetensors==0.7.0
!pip install bitsandbytes==0.49.0
!pip install peft==0.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 141.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvtx-cu12
    F

In [4]:
# Verify installations
!pip show torch transformers gradio langdetect sentencepiece huggingface-hub accelerate safetensors peft bitsandbytes

Name: torch
Version: 2.9.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, fastai, peft, torchdata
---
Name: transformers
Version: 4.57.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributor

In [5]:
# Check CUDA availability and version
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")

CUDA Available: True
CUDA Version: 12.8


## Restart the Runtime. Then execute the code below.

In [6]:
# clear cache
!rm -rf ~/.cache/huggingface

In [7]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")

PyTorch Version: 2.9.0+cu128
CUDA Available: True
CUDA Version: 12.8


In [8]:
# 1. Import libraries
import torch
import gradio as gr
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from langdetect import detect, LangDetectException
import re
import logging
import traceback
from huggingface_hub import login
from google.colab import userdata

In [9]:
# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [10]:
# Authenticate
huggingface_token = userdata.get('HF_TOKEN')
login(token=huggingface_token)

In [11]:
# 2. Configuration for LoRA Merged Model
# NOTE: Using the Merged repository ID for standalone portability
LORA_HUB_ID = "abhinandansamal/nllb-200-distilled-600M-LoRA-finetuned-odia-german-bidirectional"
ODIA_LANG_CODE = "ory_Orya"
GERMAN_LANG_CODE = "deu_Latn"

# Task Prefixes from your fine-tuning phase
PREFIX_ORI_TO_DEU = "translate Odia to German: "
PREFIX_DEU_TO_ORI = "translate German to Odia: "

In [12]:
# 3. Model Loading (Optimized for LoRA Research Specs)
print("🚀 Loading the Standalone Merged LoRA Model...")

try:
    # 4-bit Quantization (NF4): Matches your best LoRA benchmarking environment
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(LORA_HUB_ID)

    # Load the merged standalone model
    model = AutoModelForSeq2SeqLM.from_pretrained(
        LORA_HUB_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )

    # Standard Research Inference Parameters
    # Syncing with max_length=512 and num_beams=5 for optimal quality
    translator = pipeline(
        "translation",
        model=model,
        tokenizer=tokenizer,
        max_length=512,
        num_beams=5,
        length_penalty=1.0
    )
    print("✅ LoRA Merged Model loaded successfully!")
except Exception as e:
    print(f"❌ Initialization Error: {traceback.format_exc()}")
    translator = None

🚀 Loading the Standalone Merged LoRA Model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/836 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Device set to use cuda:0


✅ LoRA Merged Model loaded successfully!


In [17]:
# ==========================================
# 4. TRANSLATION LOGIC WITH PREFIX INJECTION
# ==========================================
def is_odia_script(text):
    """
    Checks if the input text contains characters from the Odia Unicode block.

    This acts as a fast heuristic for language detection, specifically targeting
    the Odia script range (U+0B00 to U+0B7F).

    Args:
        text (str): The input string to check.

    Returns:
        bool: True if at least one Odia character is detected, False otherwise.
    """
    if not text: return False

    # Compile regex pattern for the specific Odia Unicode range.
    odia_pattern = re.compile(r'[\u0B00-\u0B7F]')

    # Return True if a match is found in the text.
    return bool(odia_pattern.search(text))

def translate_logic(input_text, source_lang="auto"):
    """
    Orchestrates the translation process using the merged LoRA model.

    This function handles:
    1. Language detection (automatic or manual).
    2. Construction of the model-specific prompt (injecting the correct task prefix).
    3. Execution of the translation inference pipeline.

    Args:
        input_text (str): The sentence to be translated.
        source_lang (str, optional): The source language code ('auto', 'or', 'de').
            Defaults to "auto".

    Returns:
        str: The translated text, or an error message if detection or inference fails.
    """
    # 1. Validation: Ensure model is loaded and input is valid.
    if translator is None:
        return "Error: Model not loaded."
    if not input_text.strip():
        return "Error: Input text is empty."

    try:
        # Step A: Language Detection
        if source_lang == "auto":
            # Heuristic check: If Odia script is present, assume Odia.
            if is_odia_script(input_text):
                detected = "or"
            else:
                # Fallback: Use 'langdetect' for Latin script (German detection).
                try:
                    detected = detect(input_text)
                except:
                    return "Error: Auto-detection failed. Select language manually."
        else:
            detected = source_lang

        # Step B: Apply Prefixes and Translate
        # The model was fine-tuned with specific prefixes (e.g., "translate Odia to German: ").
        # We must reinject these prefixes at inference time to activate the LoRA adapters correctly.
        if detected == "or":
            # LoRA specialized prefix
            full_prompt = PREFIX_ORI_TO_DEU + input_text
            res = translator(full_prompt, src_lang=ODIA_LANG_CODE, tgt_lang=GERMAN_LANG_CODE)
            logger.info("LoRA Merged translating: Odia -> German")

        elif detected == "de":
            full_prompt = PREFIX_DEU_TO_ORI + input_text
            res = translator(full_prompt, src_lang=GERMAN_LANG_CODE, tgt_lang=ODIA_LANG_CODE)
            logger.info("LoRA Merged translating: German -> Odia")

        else:
            return f"Error: Language '{detected}' is not supported."

        # Extract the translation string from the pipeline response.
        return res[0]["translation_text"]

    except Exception as e:
        logger.error(f"LoRA Translation error: {e}")
        return f"Error: {str(e)}"

In [18]:
# ==========================================
# 5. GRADIO UI SETUP
# ==========================================

# UI configuration constants
title = "🚀 LoRA Merged Odia-German Translator"
description = """
### Low-Rank Adaptation (Merged)
This application uses the **LoRA Fine-Tuned NLLB-200 (600M)** model where weights have been merged for standalone inference.
* **Optimization:** Loaded in 4-bit NF4 precision.
* **Accuracy:** Uses a Beam Search width of 5.
"""

# Example queries for user convenience
examples = [
    ["ଆଜି ପାଗ ବହୁତ ଭଲ ଅଛି।", "or"],   # "The weather is very good today."
    ["Die Feuerwehr musste zahlreiche Menschen mit Booten in Sicherheit bringen.", "de"],   # Complex German sentence
    ["ମନ୍ତ୍ରୀ ଘୋଷଣା କଲେ ଯେ ଏହି ନୂଆ ରାଜପଥ ଆସନ୍ତା ବର୍ଷ ସୁଦ୍ଧା ସମ୍ପୂର୍ଣ୍ଣ ହେବ।", "or"]    # Complex Odia sentence
]

# Initialize the Gradio Interface
iface = gr.Interface(
    fn=translate_logic,
    inputs=[
        gr.Textbox(lines=4, label="Input text", placeholder="Odia or German..."),
        gr.Radio(choices=["auto", "or", "de"], label="Language Selector", value="auto")
    ],
    outputs=gr.Textbox(lines=4, label="LoRA Translation"),
    title=title,
    description=description,
    examples=examples,
    theme=gr.themes.Soft(), # A softer, modern visual theme
    allow_flagging="never"  # Disable the "Flag" button to keep the UI clean
)

# Launch the app
if __name__ == "__main__":
    # share=True creates a public URL for external testing
    iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff1490f7d19c04718c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
